In [1]:
import os
import shutil
import random
import glob
from PIL import Image

# change data format and divide data 70/30

In [1]:
import os
import random
import shutil

def split_dataset(data_dir, output_dir="train+", train_ratio=0.7):
    classes = ["Normal", "Lesion"]

    # Create BUSI root folder
    os.makedirs(output_dir, exist_ok=True)

    for cls in classes:
        class_dir = os.path.join(data_dir, cls)
        assert os.path.exists(class_dir), f"{cls} directory not found!"

        # Get files
        files = [f for f in os.listdir(class_dir)
                 if os.path.isfile(os.path.join(class_dir, f))]
        random.shuffle(files)

        split_idx = int(len(files) * train_ratio)
        train_files = files[:split_idx]
        test_files = files[split_idx:]

        # Create new directories inside BUSI
        train_dir = os.path.join(output_dir, "train", cls)
        test_dir = os.path.join(output_dir, "val", cls)

        os.makedirs(train_dir, exist_ok=True)
        os.makedirs(test_dir, exist_ok=True)

        # Copy training files
        for f in train_files:
            shutil.copy(
                os.path.join(class_dir, f),
                os.path.join(train_dir, f)
            )

        # Copy test files
        for f in test_files:
            shutil.copy(
                os.path.join(class_dir, f),
                os.path.join(test_dir, f)
            )

        print(f"{cls}: {len(train_files)} train, {len(test_files)} test")

    print(f"\nDataset successfully split into '{output_dir}' folder.")


# Example usage
if __name__ == "__main__":
    data_dir = "train"   # original dataset
    split_dataset(data_dir, output_dir="train+", train_ratio=0.7)

Normal: 84 train, 37 test
Lesion: 60 train, 27 test

Dataset successfully split into 'train+' folder.


# 5 fold with 70/30

In [1]:
import os
import random
import shutil

def split_dataset_5fold(data_dir, output_dir="breast_lesion_5folds", k=5, seed=42):

    random.seed(seed)
    classes = ["Normal", "Benign", "Malignant"]

    os.makedirs(output_dir, exist_ok=True)

    # Collect files per class
    class_files = {}

    for cls in classes:
        class_dir = os.path.join(data_dir, cls)
        assert os.path.exists(class_dir), f"{cls} directory not found!"

        files = [f for f in os.listdir(class_dir)
                 if os.path.isfile(os.path.join(class_dir, f))]

        random.shuffle(files)
        class_files[cls] = files

    # Generate folds
    for fold in range(k):

        print(f"\nCreating Fold {fold+1}")

        for cls in classes:

            files = class_files[cls]
            fold_size = len(files) // k

            start = fold * fold_size
            end = (fold + 1) * fold_size if fold < k - 1 else len(files)

            val_files = files[start:end]
            train_files = files[:start] + files[end:]

            train_dir = os.path.join(output_dir, f"fold{fold+1}", "train", cls)
            val_dir = os.path.join(output_dir, f"fold{fold+1}", "val", cls)

            os.makedirs(train_dir, exist_ok=True)
            os.makedirs(val_dir, exist_ok=True)

            # Copy train files
            for f in train_files:
                shutil.copy(
                    os.path.join(data_dir, cls, f),
                    os.path.join(train_dir, f)
                )

            # Copy validation files
            for f in val_files:
                shutil.copy(
                    os.path.join(data_dir, cls, f),
                    os.path.join(val_dir, f)
                )

            print(f"{cls}: {len(train_files)} train, {len(val_files)} val")


    print(f"\n5-fold dataset created in '{output_dir}'")


# Example usage
if __name__ == "__main__":
    data_dir = "breast_lesion"   # original dataset
    split_dataset_5fold(data_dir, output_dir="breast_lesion_5folds", k=5)


Creating Fold 1
Normal: 146 train, 36 val
Benign: 37 train, 9 val
Malignant: 28 train, 6 val

Creating Fold 2
Normal: 146 train, 36 val
Benign: 37 train, 9 val
Malignant: 28 train, 6 val

Creating Fold 3
Normal: 146 train, 36 val
Benign: 37 train, 9 val
Malignant: 28 train, 6 val

Creating Fold 4
Normal: 146 train, 36 val
Benign: 37 train, 9 val
Malignant: 28 train, 6 val

Creating Fold 5
Normal: 144 train, 38 val
Benign: 36 train, 10 val
Malignant: 24 train, 10 val

5-fold dataset created in 'breast_lesion_5folds'
